# Gemma 4 E4B — hello world

Smoke test for the fine-tuning box. In order: environment and GPU, a pinned download that writes a provenance manifest, a text-only load, one plain generation, how the tokenizer splits option-quote strings, and a zero-shot JSON extraction as a pre-fine-tuning baseline.

Nothing in this notebook touches real chat data. The quote strings below are made up.

Before running on an ephemeral box, point the Hub cache at persistent storage, e.g. `export HF_HOME=/workspace/hf` in the shell that launches Jupyter.

In [ ]:
import sys, platform
import torch, transformers, huggingface_hub

print("python      ", sys.version.split()[0], platform.machine())
print("torch       ", torch.__version__)
print("transformers", transformers.__version__)
print("hf_hub      ", huggingface_hub.__version__)

if torch.cuda.is_available():
    device = "cuda"
    p = torch.cuda.get_device_properties(0)
    print(f"GPU          {p.name}, {p.total_memory / 1e9:.0f} GB")
elif torch.backends.mps.is_available():
    device = "mps"  # Mac fallback for this notebook; mlx-lm is the real path there
    print("device       Apple Silicon (MPS)")
else:
    device = "cpu"
    print("device       CPU — loading a 16 GB model here will be slow")

## 1. Download, pinned to a commit

`main` is resolved to a commit SHA once, and every weight shard's Hub-reported SHA-256 goes into `provenance.json`. Reuse `REVISION` in the training config so every run trains against the same bytes.

In [ ]:
import json
from huggingface_hub import model_info, snapshot_download

REPO = "google/gemma-4-E4B-it"
LOCAL_DIR = "./gemma-4-E4B-it"

info = model_info(REPO, files_metadata=True)
REVISION = info.sha
path = snapshot_download(
    REPO,
    revision=REVISION,
    local_dir=LOCAL_DIR,
    allow_patterns=["*.safetensors", "*.json", "*.jinja", "*.model", "*.txt"],
)

manifest = {
    "repo": REPO,
    "revision": REVISION,
    "local_path": path,
    "sha256": {
        s.rfilename: s.lfs.sha256
        for s in info.siblings
        if s.lfs and s.rfilename.endswith(".safetensors")
    },
}
with open("provenance.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("revision:", REVISION)
print("shards:  ", len(manifest["sha256"]))

## 2. Load text-only

The repo is multimodal. Loading through `AutoModelForCausalLM` takes the language model and leaves the vision and audio towers behind — expect a warning that some weights were not used. That is the point.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained(LOCAL_DIR)
model = AutoModelForCausalLM.from_pretrained(LOCAL_DIR, dtype=torch.bfloat16, device_map=device)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"{type(model).__name__}: {n_params / 1e9:.2f}B parameters, {model.dtype}")
if device == "cuda":
    print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## 3. Chat template and the thinking switch

E4B-it has thinking modes. For extraction you want the JSON immediately, so find out how the template turns thinking off before you build training examples around it.

In [ ]:
tpl = tok.chat_template or ""
print("chat template length:", len(tpl))
hits = [kw for kw in ("enable_thinking", "thinking", "think", "reasoning") if kw in tpl]
if hits:
    print("template mentions:", hits, "— read the template and confirm the off switch")
else:
    print("no thinking switch found in the template text")

In [ ]:
def chat(prompt, max_new_tokens=64):
    msgs = [{"role": "user", "content": prompt}]
    # enable_thinking=False is ignored silently if the template has no such variable —
    # adjust once the previous cell tells you what the switch is actually called.
    text = tok.apply_chat_template(
        msgs, add_generation_prompt=True, tokenize=False, enable_thinking=False
    )
    # the template already inserts <bos>; add_special_tokens=False avoids a second one
    inputs = tok(text, return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0, inputs["input_ids"].shape[-1]:], skip_special_tokens=True)


print(chat("Say hello in one short sentence."))

## 4. How the tokenizer splits quote strings

The thing to look at is digits: one token per digit means exact copying of strikes and prices is easy for the model; multi-digit chunks mean it is copying opaque tokens. Made-up examples only.

In [ ]:
samples = [
    "SPX Dec26 4500 C 12.5/13.0 x500 ref 4480",
    "AAPL Jan27 150/170 CS 2.10 bid 5k",
    "NVDA Mar 95 P 3.25 / 3.40, 2k up, ref 101.5",
    "same, now 12.75/13.25",
    "off",
]
for s in samples:
    ids = tok(s, add_special_tokens=False)["input_ids"]
    pieces = tok.convert_ids_to_tokens(ids)
    print(f"{len(pieces):3d} tokens | {s}")
    print("    ", pieces)

## 5. Zero-shot JSON extraction

A pre-fine-tuning baseline, greedy decoding. Expect it to be roughly right and inconsistently formatted — the fine-tune fixes the formatting and the edge cases, not the basic idea.

In [ ]:
SCHEMA = (
    "Extract every option quote from the message as a JSON list. Fields: "
    "ticker, expiry (YYYY-MM-DD), strike, put_call, structure, ref_price, bid, ask, size, quote_intent. "
    "Use null for anything not stated. Return only JSON, no prose."
)
msg = "SPX Dec26 4500 C 12.5/13.0 x500 ref 4480"
print(chat(f"{SCHEMA}\n\nMessage: {msg}", max_new_tokens=200))

In [ ]:
if device == "cuda":
    print(f"peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

## Next

If every cell above runs, the box is ready. The next notebook is the LoRA fine-tune, and it starts from `REVISION` and `provenance.json` written here — not from a fresh `main`.